In [ ]:
run_id = "00000000-0000-0000-0000-000000000000"
bucket = "django-prefect-datalake-dev"
aws_access_key_id = "rustfs"
aws_secret_access_key = "rustfs_secret"
aws_s3_region = "us-east-1"
s3_endpoint = "localhost:9000"
notebook_output_dir = "data/notebook_outputs"

In [ ]:
import json
import s3fs
import polars as pl

In [ ]:
endpoint_url = f"http://{s3_endpoint}" if not s3_endpoint.startswith("http") else s3_endpoint
storage_options = {
    "key": aws_access_key_id,
    "secret": aws_secret_access_key,
    "endpoint_url": endpoint_url,
}
s3 = s3fs.S3FileSystem(key=aws_access_key_id, secret=aws_secret_access_key,
                       endpoint_url=endpoint_url,
                       client_kwargs={"region_name": aws_s3_region})

In [ ]:
input_path = f"s3://{bucket}/processed/flows/data-processing/{run_id}/03_transformed.parquet"
df = pl.scan_parquet(input_path, storage_options=storage_options)

In [ ]:
result_df = (
    df
    .group_by(['year_month', 'amount_category'])
    .agg([
        pl.col('total').sum().alias('total_revenue'),
        pl.col('id').count().alias('transaction_count'),
        pl.col('customer_id').n_unique().alias('unique_customers'),
        pl.col('total').mean().alias('avg_transaction_value'),
    ])
    .sort(['year_month', 'total_revenue'], descending=[False, True])
    .collect()
)

In [ ]:
s3_output_path = f"processed/flows/data-processing/{run_id}/output.parquet"
output_full = f"s3://{bucket}/{s3_output_path}"

with s3.open(output_full, "wb") as f:
    result_df.write_parquet(f, compression="snappy", statistics=True, use_pyarrow=True)

row_count = len(result_df)
file_size_mb = round(s3.info(output_full)["size"] / (1024 * 1024), 2)

print(f"Aggregated {row_count} rows → {output_full}")

In [ ]:
# IMPORTANT: This JSON is parsed by PipelineRunner._extract_metadata()
# Must be the last printed line and valid JSON.
print(json.dumps({
    "row_count": row_count,
    "column_count": len(result_df.columns),
    "columns": result_df.columns,
    "s3_output_path": s3_output_path,
    "file_size_mb": file_size_mb,
}))